# Intermediate 01: Identity Propagation and Delegated Authority

In simple systems, knowing *who is calling* is easy. In agentic architectures, a request might hop from the User -> Agent -> Document Service -> Storage Service.

If the Agent uses its own powerful infrastructure privileges to fetch data, it can be tricked into retrieving documents the original User isn't allowed to see. This is the **Confused Deputy** problem.

The solution is **Identity Propagation**: securely passing the user's identity and a tightly scoped **Delegation Grant** all the way down the chain.

In [ ]:
import sys
from pathlib import Path
import importlib
sys.path.append(str(Path.cwd().parent.parent.parent / 'curriculum' / 'intermediate' / '01-identity-propagation'))
lab = importlib.import_module('01_identity_propagation')
from datetime import datetime, timezone, timedelta

clock = lambda: datetime(2025, 1, 1, 12, 0, tzinfo=timezone.utc)
ds = lab.DelegationService(clock_fn=clock)
audit = lab.AuditSink()
storage = lab.StorageService(ds, audit)
secure_docs = lab.SecureDocumentService(ds, storage, audit)
naive_docs = lab.NaiveDocumentService(storage)
app = lab.ResearchApplication(ds, secure_docs, naive_docs, audit)


## 1. The Confused Deputy (Ambient Authority)

First, let's see how a naive service behaves. The `NaiveDocumentService` trusts the Research Agent's workload identity completely. It uses its *ambient* service-level authority to query storage, ignoring the user.

Alice only has access to `doc-101` and `doc-102`. But watch what happens when she asks for `doc-secret` through the naive API:

In [ ]:
ans = app.answer_naive("alice", "doc-secret")
print(f"Result: {ans}")

Because the downstream service only saw `research-agent`, it happily returned the highly classified acquisition plans. The agent was tricked into bypassing Alice's restrictions.

## 2. Secure Identity Propagation

Now, let's use the secure API. Here, the `ResearchApplication` issues a **Delegation Grant** to the agent. This grant binds Alice (Principal) to the Agent (Delegate), restricted to `doc-secret`.

However, the Issuer will enforce *monotonic down-scoping*: it will **refuse to issue a grant for a resource Alice does not already possess**.

In [ ]:
r_allowed = app.answer_secure("alice", "doc-101", "req-allowed")
r_denied = app.answer_secure("alice", "doc-secret", "req-denied")

print(f"Doc-101 (Allowed): {r_allowed.answer}")
print(f"Doc-Secret (Denied): {r_denied.answer}")

## 3. Delegation Audit Trail

Let's inspect the secure multi-hop audit trail for `req-allowed`. Notice how both the Workload Identity (who is acting) and the Audience (who they are talking to) shift at each hop, while Alice (Principal) is preserved throughout.

In [ ]:
events = [e for e in audit.events if e.correlation_id == "req-allowed"]
for e in events:
    print(f"Hop {e.delegation_depth}:")
    print(f"  Principal: {e.principal_id}")
    print(f"  Delegate:  {e.workload_id}")
    print(f"  Audience:  {e.audience}")
    print(f"  Action:    {e.operation} {e.resource_id}")
    print(f"  Decision:  {e.decision} ({e.reason})\n")

## 4. Audience Restriction and Scope

If an attacker steals a token meant for the Document Service and tries to replay it against the Storage Service directly, it will fail due to **Audience Mismatch**. If they try to modify a document with a read-only token, it fails due to **Scope Mismatch**.

In [ ]:
alice = lab.PRINCIPAL_REGISTRY["alice"]
agent = lab.WORKLOAD_REGISTRY["research-agent"]

# 1. Create a grant intended for the Document Service
grant = ds.issue(alice, agent, audience="document-service", requested_operations={"read"}, requested_resources={"doc-101"})

# 2. Attempt to use it directly against Storage Service (Wrong Audience)
res = storage.read_object("research-agent", grant, "doc-101", "req-steal")
print(f"Storage response: {res}")
print(f"Audit Reason: {audit.events[-1].reason}")